In [1]:

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
import yfinance as yf

from dotenv import load_dotenv
import os
import sys

import sqlite3

load_dotenv()  # carga .env
sys.path.insert(0, os.path.join(os.environ["CUANTIS_ROOT"], "src"))

In [4]:
# Utils from cuantis_utils
from cuantis_utils.get_prices_options import resolve_db_path, read_option_chain_history
from cuantis_utils.get_prices_options import OptionChainDownloader

In [5]:
downloader = OptionChainDownloader()
result = downloader.fetch_snapshot(ticker="AAPL")

In [ ]:
result.groupby(by = "expiration_date")["contract_symbol"].¿¿

expiration_date
2026-02-13    112
2026-02-18     63
2026-02-20    144
2026-02-23     49
2026-02-25     30
2026-02-27     99
2026-03-06     79
2026-03-13     59
2026-03-20    117
2026-03-27     57
2026-04-17    106
2026-05-15    144
2026-06-18    155
2026-07-17     93
2026-08-21    111
2026-09-18    144
2026-11-20    106
2026-12-18    160
2027-01-15    136
2027-06-17    134
2027-12-17    174
2028-01-21    124
2028-03-17     98
2028-12-15    110
Name: contract_symbol, dtype: int64

In [8]:
db_path = resolve_db_path('src/data/options/option_chain_history.sqlite', must_exist=True)
df_all = read_option_chain_history(db_path=db_path)
df_all

,ticker,snapshot_utc,expiration_date,option_type,contract_symbol,last_trade_date_utc,strike,last_price,bid,ask,...,percent_change,volume,open_interest,implied_volatility,in_the_money,contract_size,currency,underlying_price,source,downloaded_at_utc
0,CVX,2026-02-14,2026-02-20,call,CVX260220C00075000,2026-02-13,75.0,109.16,106.55,110.40,...,2.632572,5.0,2.0,4.142583,1,REGULAR,USD,183.74,yfinance,2026-02-14T05:09:44.660680+00:00
1,CVX,2026-02-14,2026-02-20,call,CVX260220C00095000,2026-02-13,95.0,89.20,86.50,90.40,...,26.327711,5.0,1.0,3.169924,1,REGULAR,USD,183.74,yfinance,2026-02-14T05:09:44.660680+00:00
2,CVX,2026-02-14,2026-02-20,call,CVX260220C00110000,2026-02-13,110.0,74.16,71.55,75.40,...,2.714691,40.0,7.0,2.565433,1,REGULAR,USD,183.74,yfinance,2026-02-14T05:09:44.660680+00:00
3,CVX,2026-02-14,2026-02-20,call,CVX260220C00115000,2026-02-13,115.0,69.05,66.55,70.40,...,31.724543,195.0,26.0,2.380375,1,REGULAR,USD,183.74,yfinance,2026-02-14T05:09:44.660680+00:00
4,CVX,2026-02-14,2026-02-20,call,CVX260220C00120000,2026-02-13,120.0,62.73,61.55,65.40,...,14.408166,2350.0,67.0,2.202641,1,REGULAR,USD,183.74,yfinance,2026-02-14T05:09:44.660680+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3182,CVX,2026-02-12,2028-12-15,put,CVX281215P00185000,2026-02-12,185.0,32.00,28.90,31.45,...,9.589039,3.0,13.0,0.247200,1,REGULAR,USD,182.79,yfinance,2026-02-12T17:52:35.345418+00:00
3183,CVX,2026-02-12,2028-12-15,put,CVX281215P00190000,2026-02-12,190.0,32.20,31.40,33.90,...,1.098902,12.0,145.0,0.242470,1,REGULAR,USD,182.79,yfinance,2026-02-12T17:52:35.345418+00:00
3184,CVX,2026-02-12,2028-12-15,put,CVX281215P00195000,2026-02-11,195.0,34.44,33.55,36.75,...,0.000000,4.0,2.0,0.240059,1,REGULAR,USD,182.79,yfinance,2026-02-12T17:52:35.345418+00:00
3185,CVX,2026-02-12,2028-12-15,put,CVX281215P00220000,2026-02-02,220.0,56.14,49.00,53.70,...,0.000000,NaN,5.0,0.236427,1,REGULAR,USD,182.79,yfinance,2026-02-12T17:52:35.345418+00:00


In [7]:
df_all.info()

<class 'pandas.DataFrame'>
RangeIndex: 1630 entries, 0 to 1629
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ticker               1630 non-null   str    
 1   snapshot_utc         1630 non-null   str    
 2   expiration_date      1630 non-null   str    
 3   option_type          1630 non-null   str    
 4   contract_symbol      1630 non-null   str    
 5   last_trade_date_utc  1630 non-null   str    
 6   strike               1630 non-null   float64
 7   last_price           1630 non-null   float64
 8   bid                  1630 non-null   float64
 9   ask                  1630 non-null   float64
 10  change               1630 non-null   float64
 11  percent_change       1630 non-null   float64
 12  volume               1528 non-null   float64
 13  open_interest        1630 non-null   float64
 14  implied_volatility   1630 non-null   float64
 15  in_the_money         1630 non-null   int64  
 16 

In [9]:
ts = pd.to_datetime(df_all["downloaded_at_utc"], utc=True, errors="coerce")
ultima = ts.max()

print("Última descarga (UTC):", ultima)
print("Última descarga (CDMX):", ultima.tz_convert("America/Mexico_City"))

Última descarga (UTC): 2026-02-14 05:09:44.660680+00:00
Última descarga (CDMX): 2026-02-13 23:09:44.660680-06:00
